# Parallel Image Processing in Python
### Gaussian Blur + Sobel Edge Detection | Sequential vs Parallel CPU

**Course project:** Parallel / Distributed Systems  
**Platform:** Kaggle Notebooks (CPU) | Also works in Jupyter and Google Colab

**What this notebook does:**
1. Generates a sample image automatically (no upload is required).
2. Implements **Gaussian blur** and **Sobel edge detection** from pixel-level loops.
3. Implements both **sequential (`range`)** and **parallel (`numba.prange`)** versions.
4. Uses multiple CPU threads; Numba can use **OpenMP (`omp`)** if supported by the runtime.
5. Checks that parallel and sequential results are **pixel-identical**.
6. Benchmarks several thread counts, graphs execution time and speedup, and saves files.

> **Instructor requirement:** The project idea specifies OpenMP or CUDA. This notebook uses Python + Numba's CPU parallelization. It will report which Numba threading backend actually ran. If the doctor *strictly* requires OpenMP, verify that the displayed backend is `omp` (otherwise do not claim OpenMP was used). CUDA is **not** used.

## 1. Setup

Use a **CPU notebook**. On Kaggle choose **New Notebook → Accelerator: None → Run All**.

Kaggle commonly includes these libraries. If an import fails, install the missing package in a separate cell using e.g. `%pip install numba Pillow matplotlib` and restart the notebook kernel.

For fair timing, the parallel loops and the sequential loops are **both compiled to native machine code** using Numba. We exclude initial compilation from the benchmarks.

In [ ]:
import os
# Prefer OpenMP if it exists; otherwise permit Numba's other supported backends.
# This must be set BEFORE importing numba.
os.environ.setdefault("NUMBA_THREADING_LAYER_PRIORITY", "omp tbb workqueue")

import csv
import json
import math
import statistics
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from numba import njit, prange, set_num_threads, get_num_threads, threading_layer, config

OUTPUT = Path('/kaggle/working/parallel_image_results') if Path('/kaggle/working').exists() else Path('parallel_image_results')
OUTPUT.mkdir(parents=True, exist_ok=True)

SIGMA = 2.0                    # How much Gaussian blur to apply
REPEATS = 3                    # Timing repetitions; use 5 for final report
REQUESTED_THREAD_COUNTS = [1, 2, 4, 8]
THREAD_COUNTS = [n for n in REQUESTED_THREAD_COUNTS if n <= int(config.NUMBA_NUM_THREADS)]
if not THREAD_COUNTS:
    THREAD_COUNTS = [1]
DEMO_THREADS = max(THREAD_COUNTS)

print('Numba maximum available threads:', config.NUMBA_NUM_THREADS)
print('Thread counts for benchmark:', THREAD_COUNTS)
print('Results folder:', OUTPUT.resolve())

## 2. Load an image

The cell below creates an **original sample cityscape**, so the notebook works immediately.

To use **your own image** on Kaggle: upload the image as a Kaggle Dataset with **Add Input**, then change `IMAGE_PATH` to the full path (such as `/kaggle/input/my-images/photo.jpg`). Alternatively leave `IMAGE_PATH = None` to use the included sample.

In [ ]:
def generate_sample(destination: Path, width: int = 1200, height: int = 800) -> Path:
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    yy = np.linspace(0, 1, height, dtype=np.float32)[:, None]
    xx = np.linspace(0, 1, width, dtype=np.float32)[None, :]
    data = np.empty((height, width, 3), dtype=np.uint8)
    data[:, :, 0] = np.clip(43 + 140 * yy + 24 * xx, 0, 255).astype(np.uint8)
    data[:, :, 1] = np.clip(83 + 64 * yy + 9 * xx, 0, 255).astype(np.uint8)
    data[:, :, 2] = np.clip(155 - 88 * yy + 6 * xx, 0, 255).astype(np.uint8)
    im = Image.fromarray(data, 'RGB')
    dr = ImageDraw.Draw(im)
    dr.ellipse((800, 65, 1010, 275), fill='#fef3bb', outline='#ffe8a3', width=5)
    dr.polygon([(0, 560), (160, 400), (315, 570), (485, 320), (700, 550), (850, 430), (1200, 595), (1200, 800), (0, 800)], fill='#314f68')
    dr.polygon([(0, 660), (180, 575), (320, 630), (550, 470), (780, 670), (960, 560), (1200, 620), (1200, 800), (0, 800)], fill='#244457')
    # Contrasting edges and detailed buildings make Sobel output easy to explain.
    buildings = [(60, 420, 180, 715), (190, 495, 330, 715), (360, 405, 485, 715),
                 (510, 540, 650, 715), (690, 360, 830, 715), (870, 460, 975, 715),
                 (995, 390, 1160, 715)]
    for idx, (x1, y1, x2, y2) in enumerate(buildings):
        dr.rectangle((x1, y1, x2, y2), fill=('#172b41' if idx % 2 else '#203b54'), outline='#77adc2', width=3)
        for wy in range(y1 + 24, y2 - 9, 32):
            for wx in range(x1 + 16, x2 - 10, 26):
                dr.rectangle((wx, wy, wx + 11, wy + 13), fill=('#f7d686' if ((wx + wy) // 13) % 3 else '#648ea9'))
    dr.rectangle((0, 716, width, height), fill='#162c3b')
    dr.line((0, 728, width, 728), fill='#f6c77a', width=6)
    for px in range(0, width, 110):
        dr.rectangle((px + 25, 766, px + 86, 771), fill='#e6ecf3')
    dr.rounded_rectangle((40, 32, 590, 152), radius=18, fill='#0f263b', outline='#a1d8da', width=3)
    dr.text((67, 50), 'PARALLEL IMAGE PROCESSING', fill='#ffffff', font=ImageFont.truetype('DejaVuSans-Bold.ttf', 28) if _font_exists() else None)
    dr.text((69, 104), 'Gaussian Blur  |  Sobel Edge Detection', fill='#b4e5eb')
    im.save(destination)
    return destination


def _font_exists():
    try:
        ImageFont.truetype('DejaVuSans-Bold.ttf', 28)
        return True
    except OSError:
        return False


SAMPLE_PATH = OUTPUT / 'sample_image.png'
generate_sample(SAMPLE_PATH)

IMAGE_PATH = os.environ.get("BENCHMARK_IMAGE", "sample-image.png")
selected_image = Path(IMAGE_PATH) if IMAGE_PATH else SAMPLE_PATH
if not selected_image.is_file():
    raise FileNotFoundError(f'Image not found: {selected_image}')

image = np.asarray(Image.open(selected_image).convert('RGB'), dtype=np.uint8)
print('Image:', selected_image, '| dimensions:', image.shape[1], 'x', image.shape[0])
plt.figure(figsize=(11, 6))
plt.imshow(image)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 3. The image processing algorithms

**Gaussian blur:** Uses a 1-D Gaussian kernel twice: one horizontal pass and one vertical pass. The pixels at image borders are clamped to the nearest valid pixel. Each pass is parallelized independently, and threads read a shared *unchanging* input image and write to separate output pixels.

**Sobel edge detection:** Converts the image to grayscale, then computes horizontal (`Gx`) and vertical (`Gy`) gradients using the surrounding 3×3 pixels. Edge strength is `min(255, √(Gx² + Gy²))`.

**Parallel part:** `@njit(parallel=True)` combined with `prange(height)` lets Numba distribute image rows across multiple CPU threads. The sequential implementations use `@njit` and `range(height)` so both versions avoid Python interpreter overhead.

In [ ]:
# Compiled pixel-by-pixel implementations of both filters.
@njit
def _horizontal_serial(image, kernel):
    h, w, channels = image.shape
    radius = len(kernel) // 2
    out = np.empty_like(image)
    for y in range(h):
        for x in range(w):
            for c in range(channels):
                value = 0.0
                for k in range(-radius, radius + 1):
                    xx = max(0, min(w - 1, x + k))
                    value += image[y, xx, c] * kernel[k + radius]
                out[y, x, c] = value
    return out


@njit(parallel=True)
def _horizontal_parallel(image, kernel):
    h, w, channels = image.shape
    radius = len(kernel) // 2
    out = np.empty_like(image)
    for y in prange(h):
        for x in range(w):
            for c in range(channels):
                value = 0.0
                for k in range(-radius, radius + 1):
                    xx = max(0, min(w - 1, x + k))
                    value += image[y, xx, c] * kernel[k + radius]
                out[y, x, c] = value
    return out


@njit
def _vertical_serial(image, kernel):
    h, w, channels = image.shape
    radius = len(kernel) // 2
    out = np.empty_like(image)
    for y in range(h):
        for x in range(w):
            for c in range(channels):
                value = 0.0
                for k in range(-radius, radius + 1):
                    yy = max(0, min(h - 1, y + k))
                    value += image[yy, x, c] * kernel[k + radius]
                out[y, x, c] = value
    return out


@njit(parallel=True)
def _vertical_parallel(image, kernel):
    h, w, channels = image.shape
    radius = len(kernel) // 2
    out = np.empty_like(image)
    for y in prange(h):
        for x in range(w):
            for c in range(channels):
                value = 0.0
                for k in range(-radius, radius + 1):
                    yy = max(0, min(h - 1, y + k))
                    value += image[yy, x, c] * kernel[k + radius]
                out[y, x, c] = value
    return out


@njit
def _sobel_serial(gray):
    h, w = gray.shape
    out = np.empty((h, w), dtype=np.float32)
    for y in range(h):
        ym = max(0, y - 1)
        yp = min(h - 1, y + 1)
        for x in range(w):
            xm = max(0, x - 1)
            xp = min(w - 1, x + 1)
            gx = (-gray[ym, xm] + gray[ym, xp]
                  - 2.0 * gray[y, xm] + 2.0 * gray[y, xp]
                  - gray[yp, xm] + gray[yp, xp])
            gy = (-gray[ym, xm] - 2.0 * gray[ym, x] - gray[ym, xp]
                  + gray[yp, xm] + 2.0 * gray[yp, x] + gray[yp, xp])
            magnitude = math.sqrt(gx * gx + gy * gy)
            out[y, x] = min(255.0, magnitude)
    return out


@njit(parallel=True)
def _sobel_parallel(gray):
    h, w = gray.shape
    out = np.empty((h, w), dtype=np.float32)
    for y in prange(h):
        ym = max(0, y - 1)
        yp = min(h - 1, y + 1)
        for x in range(w):
            xm = max(0, x - 1)
            xp = min(w - 1, x + 1)
            gx = (-gray[ym, xm] + gray[ym, xp]
                  - 2.0 * gray[y, xm] + 2.0 * gray[y, xp]
                  - gray[yp, xm] + gray[yp, xp])
            gy = (-gray[ym, xm] - 2.0 * gray[ym, x] - gray[ym, xp]
                  + gray[yp, xm] + 2.0 * gray[yp, x] + gray[yp, xp])
            magnitude = math.sqrt(gx * gx + gy * gy)
            out[y, x] = min(255.0, magnitude)
    return out

In [ ]:
# Build the Gaussian kernel, prepare inputs, and select sequential/parallel execution.
def gaussian_kernel(sigma: float) -> np.ndarray:
    """Normalized 1-D Gaussian with radius ceil(3*sigma)."""
    sigma = float(sigma)
    if not (0.1 <= sigma <= 20) or not math.isfinite(sigma):
        raise ValueError('Sigma must be between 0.1 and 20.')
    radius = max(1, math.ceil(3 * sigma))
    offsets = np.arange(-radius, radius + 1, dtype=np.float64)
    weights = np.exp(-0.5 * (offsets / sigma) ** 2)
    return (weights / weights.sum()).astype(np.float32)


def prepare_image(image: np.ndarray, filter_name: str) -> np.ndarray:
    """Pre-process inputs identically for both algorithms (not part of timed kernels)."""
    img = np.asarray(image)
    if img.ndim == 2:
        img = np.stack((img, img, img), axis=2)
    if img.ndim != 3 or img.shape[2] not in (3, 4):
        raise ValueError('Image must be RGB, RGBA, or grayscale.')
    rgb = np.ascontiguousarray(img[:, :, :3], dtype=np.float32)
    if filter_name == 'gaussian':
        return rgb
    if filter_name == 'sobel':
        gray = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]
        return np.ascontiguousarray(gray, dtype=np.float32)
    raise ValueError('Filter must be gaussian or sobel.')


def max_threads() -> int:
    return int(config.NUMBA_NUM_THREADS)


def selected_backend() -> str:
    try:
        return threading_layer()
    except ValueError:
        return 'not initialized'


def run_filter(prepared: np.ndarray, filter_name: str,
               mode: str = 'parallel', sigma: float = 2.0,
               threads: int = 4) -> np.ndarray:
    """Return uint8 image. Each output pixel is written by one iteration only."""
    if mode not in ('parallel', 'sequential'):
        raise ValueError('Mode must be parallel or sequential.')
    if mode == 'parallel':
        if not (1 <= int(threads) <= max_threads()):
            raise ValueError(f'Threads must be in 1..{max_threads()}.')
        set_num_threads(int(threads))
    if filter_name == 'gaussian':
        if prepared.ndim != 3:
            raise ValueError('Gaussian expects an RGB prepared image.')
        kernel = gaussian_kernel(sigma)
        if mode == 'parallel':
            tmp = _horizontal_parallel(prepared, kernel)
            result = _vertical_parallel(tmp, kernel)
        else:
            tmp = _horizontal_serial(prepared, kernel)
            result = _vertical_serial(tmp, kernel)
    elif filter_name == 'sobel':
        if prepared.ndim != 2:
            raise ValueError('Sobel expects a grayscale prepared image.')
        result = (_sobel_parallel(prepared) if mode == 'parallel'
                  else _sobel_serial(prepared))
    else:
        raise ValueError('Filter must be gaussian or sobel.')
    return np.clip(np.rint(result), 0, 255).astype(np.uint8)

## 4. Run filters and preview the output

This cell shows three images: **original**, **Gaussian blur**, and **Sobel edges**. It saves the processed PNG files and verifies that parallel output equals sequential output pixel-for-pixel.

In [ ]:
results = {}
for filter_name in ('gaussian', 'sobel'):
    prepared = prepare_image(image, filter_name)
    reference = run_filter(prepared, filter_name, mode='sequential', sigma=SIGMA)
    output = run_filter(prepared, filter_name, mode='parallel', sigma=SIGMA, threads=DEMO_THREADS)
    assert np.array_equal(reference, output), f'{filter_name} has different output!'
    path = OUTPUT / f'{filter_name}_result.png'
    Image.fromarray(output).save(path)
    results[filter_name] = output
    print(f'{filter_name.title()}: pixel-identical ✅  | saved: {path.name}')

print('Actual Numba threading backend:', selected_backend())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, pic, name in zip(axes, [image, results['gaussian'], results['sobel']],
                         ['Original image', 'Gaussian blur', 'Sobel edge detection']):
    ax.imshow(pic, cmap='gray' if pic.ndim == 2 else None, vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT / 'image_comparison.png', dpi=160)
plt.show()

## 5. Fair performance measurements

We run **each method once first** to warm up the JIT compilation and **do not count** compilation time. We alternate the test order across repetitions, check correctness after each test, and report **median milliseconds**.

- **Speedup** = sequential time ÷ parallel time.
- **Parallel efficiency (%)** = speedup ÷ number of threads × 100.
- `Speedup > 1` means parallel was faster in the measured run. It is **not guaranteed**; small images and thread overhead can make parallel processing slower.

For your final results, use your own measurements rather than copying numbers from another machine.

In [ ]:
def benchmark_once(prepared, filter_name, threads, repeats=3, sigma=2.0):
    # Warm up / compile both branches OUTSIDE the timed section.
    baseline = run_filter(prepared, filter_name, 'sequential', sigma, threads)
    other = run_filter(prepared, filter_name, 'parallel', sigma, threads)
    assert np.array_equal(baseline, other), 'Outputs must be identical.'

    times = {'sequential': [], 'parallel': []}
    for run_number in range(repeats):
        order = ('sequential', 'parallel') if run_number % 2 == 0 else ('parallel', 'sequential')
        for mode in order:
            start = perf_counter()
            current = run_filter(prepared, filter_name, mode, sigma, threads)
            elapsed_ms = (perf_counter() - start) * 1000
            assert np.array_equal(current, baseline), 'Unexpected output mismatch.'
            times[mode].append(elapsed_ms)

    seq_ms = statistics.median(times['sequential'])
    par_ms = statistics.median(times['parallel'])
    speedup = seq_ms / par_ms
    return {
        'filter': filter_name,
        'width': int(prepared.shape[1]),
        'height': int(prepared.shape[0]),
        'threads': threads,
        'threading_backend': selected_backend(),
        'repeats': repeats,
        'sequential_ms': round(seq_ms, 4),
        'parallel_ms': round(par_ms, 4),
        'speedup': round(speedup, 3),
        'efficiency_percent': round(speedup / threads * 100, 2),
        'pixels_identical': True,
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    }

records = []
for name in ('gaussian', 'sobel'):
    prepared = prepare_image(image, name)
    for threads in THREAD_COUNTS:
        record = benchmark_once(prepared, name, threads, REPEATS, SIGMA)
        records.append(record)
        print(f"{name:8} | {threads:>2} thread(s) | "
              f"sequential {record['sequential_ms']:>9.3f} ms | "
              f"parallel {record['parallel_ms']:>9.3f} ms | "
              f"speedup {record['speedup']:>5.2f}x | "
              f"backend {record['threading_backend']}")

## 6. Performance graphs

The first chart compares sequential and parallel processing times. The second chart shows **speedup versus thread count** (1.0× is the break-even line). The graphs are saved as PNG files.

In [ ]:
# Timing comparison, one pair per filter/thread configuration
labels = [f"{r['filter']}\n{r['threads']}T" for r in records]
x = np.arange(len(records))
width = 0.36
fig, ax = plt.subplots(figsize=(max(9, len(records) * 1.65), 5))
ax.bar(x-width/2, [r['sequential_ms'] for r in records], width, label='Sequential')
ax.bar(x+width/2, [r['parallel_ms'] for r in records], width, label='Parallel')
ax.set_xticks(x, labels)
ax.set_ylabel('Median execution time (ms) — lower is better')
ax.set_title('Sequential vs Parallel Image Processing')
ax.legend()
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / 'execution_time_chart.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for name in ('gaussian', 'sobel'):
    group = [r for r in records if r['filter'] == name]
    ax.plot([r['threads'] for r in group], [r['speedup'] for r in group], 'o-', label=name.title())
ax.axhline(1.0, linestyle='--', color='gray', label='No speedup (1×)')
ax.set_xticks(THREAD_COUNTS)
ax.set_xlabel('Number of CPU threads')
ax.set_ylabel('Speedup = Sequential / Parallel')
ax.set_title('Parallel Processing Speedup')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / 'speedup_chart.png', dpi=160)
plt.show()

## 7. Save and download your results

The last cell exports CSV and JSON measurements and a ZIP containing the PNG previews and charts. In Kaggle, open the **Output** or **Files** browser after running the notebook and download files from `/kaggle/working/parallel_image_results/`.

In [ ]:
csv_file = OUTPUT / 'benchmark_results.csv'
json_file = OUTPUT / 'benchmark_results.json'
with csv_file.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
    writer.writeheader()
    writer.writerows(records)
with json_file.open('w', encoding='utf-8') as f:
    json.dump(records, f, indent=2)

import zipfile
zip_path = OUTPUT.parent / 'parallel_image_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print('All results saved to:', OUTPUT)
print('ZIP of outputs:', zip_path)
print('Files:')
for path in sorted(OUTPUT.iterdir()):
    print('  ', path.name)

## 8. Explanation for the project presentation

**Question: Why parallel processing?**  
We divide independent image rows among multiple CPU threads, allowing several portions of the image to be processed simultaneously.

**Question: What is the difference between sequential and parallel?**  
Sequential processing uses `range` so one execution thread handles rows in sequence. Parallel processing uses `prange` with `@njit(parallel=True)` to distribute row iterations across a CPU thread pool.

**Question: Why do you need Numba?**  
Pure Python loops are slow. Numba compiles both versions to native machine code, making the speed comparison about multithreading instead of interpreter overhead.

**Question: What are Gaussian and Sobel?**  
Gaussian smooths small details/noise with weighted averages. Sobel emphasizes intensity changes that indicate edges.

**Question: How did you verify correctness?**  
We used `np.array_equal` to confirm both versions produced identical output pixels for each filter.

**Question: Is this distributed computing?**  
Strictly speaking, this is **shared-memory CPU parallel computing on one machine**, not distributed computing across multiple computers. If your doctor requires multiple *nodes* or MPI, that is an additional requirement.

**Question: Is OpenMP guaranteed?**  
No. Read the printed **Actual Numba threading backend**. It may be `omp`, `tbb`, or `workqueue`. Only say this run used OpenMP if it actually prints `omp`. CUDA/GPU is not part of this notebook.

### Good submission checklist
- Run every cell with no errors, preferably with REPEATS = 5 for final tests.
- Include the screenshot of original / blur / edge detection.
- Include `benchmark_results.csv`, time chart, and speedup chart.
- Explain that performance depends on available CPU cores and image size.
- Confirm that Python + Numba fulfills your instructor's implementation requirement.